# Final Assignment: Part 1 - Create Visualizations using Matplotlib, Seaborn & Folium

**Project:** Analyzing the Impact of Recession on Automobile Sales

This notebook completes Tasks 1.1–1.9 from the final project. The official historical automobile sales dataset is loaded from the IBM Skills Network data URL when the notebook is executed.


## Setup
Libraries used: Pandas, NumPy, Matplotlib, Seaborn and Folium.


In [ ]:
%pip install -q pandas numpy matplotlib seaborn folium requests

import os
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

sns.set_theme(style="whitegrid")
DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/historical_automobile_sales.csv"

local_file = "historical_automobile_sales.csv"
if os.path.exists(local_file):
    df = pd.read_csv(local_file)
else:
    response = requests.get(DATA_URL, timeout=60)
    response.raise_for_status()
    df = pd.read_csv(io.BytesIO(response.content))
    df.to_csv(local_file, index=False)

df["Date"] = pd.to_datetime(df["Date"])
print("Data loaded successfully.")
print("Shape:", df.shape)
print("Columns:", list(df.columns))


## Task 1.1 — Automobile sales fluctuation from year to year


In [ ]:
yearly_sales = df.groupby("Year")["Automobile_Sales"].mean()

plt.figure(figsize=(10, 6))
yearly_sales.plot(kind="line", marker="o")
plt.xlabel("Year")
plt.ylabel("Average Automobile Sales")
plt.title("Automobile Sales over Time")
plt.tight_layout()
plt.savefig("Task_1_1_Automobile_Sales_over_Time.png", dpi=150, bbox_inches="tight")
plt.show()


## Task 1.2 — Advertising expenditure and automobile sales during non-recession periods


In [ ]:
non_recession = df[df["Recession"] == 0]
ad_sales = non_recession.groupby("Year", as_index=False)[
    ["Advertising_Expenditure", "Automobile_Sales"]
].mean()

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(ad_sales["Year"], ad_sales["Advertising_Expenditure"],
         marker="o", label="Advertising Expenditure")
ax1.set_xlabel("Year")
ax1.set_ylabel("Advertising Expenditure")

ax2 = ax1.twinx()
ax2.plot(ad_sales["Year"], ad_sales["Automobile_Sales"],
         marker="s", label="Automobile Sales")
ax2.set_ylabel("Average Automobile Sales")

plt.title("Advertising Expenditure and Automobile Sales during Non-Recession Periods")
fig.tight_layout()
plt.savefig("Task_1_2_Advertising_vs_Sales_Non_Recession.png", dpi=150, bbox_inches="tight")
plt.show()


## Task 1.3 — Vehicle sales during recession vs non-recession periods


In [ ]:
vehicle_period = df.groupby(
    ["Recession", "Vehicle_Type"], as_index=False
)["Automobile_Sales"].mean()

plt.figure(figsize=(12, 6))
sns.barplot(
    data=vehicle_period,
    x="Vehicle_Type",
    y="Automobile_Sales",
    hue="Recession"
)
plt.xlabel("Vehicle Type")
plt.ylabel("Average Automobile Sales")
plt.title("Vehicle-Wise Sales during Recession and Non-Recession Periods")
plt.xticks(rotation=25)
plt.legend(title="Period", labels=["Non-Recession", "Recession"])
plt.tight_layout()
plt.savefig("Task_1_3_Vehicle_Sales_Recession_vs_Non_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.4 — GDP variations during recession and non-recession periods


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

recession_gdp = df[df["Recession"] == 1].groupby("Year")["GDP"].mean()
non_recession_gdp = df[df["Recession"] == 0].groupby("Year")["GDP"].mean()

axes[0].plot(recession_gdp.index, recession_gdp.values, marker="o")
axes[0].set_title("GDP during Recession Periods")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Average GDP")

axes[1].plot(non_recession_gdp.index, non_recession_gdp.values, marker="o")
axes[1].set_title("GDP during Non-Recession Periods")
axes[1].set_xlabel("Year")

plt.suptitle("GDP Variation: Recession vs Non-Recession")
plt.tight_layout()
plt.savefig("Task_1_4_GDP_Recession_vs_Non_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.5 — Bubble plot showing the impact of seasonality on automobile sales


In [ ]:
seasonality = df.groupby("Month", as_index=False).agg(
    Automobile_Sales=("Automobile_Sales", "mean"),
    Seasonality_Weight=("Seasonality_Weight", "mean")
)

month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
seasonality["Month"] = pd.Categorical(
    seasonality["Month"], categories=month_order, ordered=True
)
seasonality = seasonality.sort_values("Month")

plt.figure(figsize=(12, 6))
plt.scatter(
    seasonality["Month"],
    seasonality["Automobile_Sales"],
    s=seasonality["Seasonality_Weight"].clip(lower=0.05) * 700,
    alpha=0.6
)
plt.xlabel("Month")
plt.ylabel("Average Automobile Sales")
plt.title("Impact of Seasonality on Automobile Sales")
plt.tight_layout()
plt.savefig("Task_1_5_Seasonality_Bubble_Plot.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.6 — Correlation between average vehicle price and sales during recessions


In [ ]:
recession_data = df[df["Recession"] == 1]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=recession_data,
    x="Price",
    y="Automobile_Sales",
    hue="Vehicle_Type",
    alpha=0.75
)
plt.xlabel("Average Vehicle Price")
plt.ylabel("Automobile Sales")
plt.title("Average Vehicle Price vs Automobile Sales during Recessions")
plt.tight_layout()
plt.savefig("Task_1_6_Price_vs_Sales_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.7 — Portion of advertising expenditure during recession and non-recession periods


In [ ]:
ad_period = df.groupby("Recession")["Advertising_Expenditure"].sum()

plt.figure(figsize=(7, 7))
plt.pie(
    ad_period.values,
    labels=["Non-Recession", "Recession"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Advertising Expenditure: Recession vs Non-Recession")
plt.tight_layout()
plt.savefig("Task_1_7_Advertising_Expenditure_Recession_vs_Non_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.8 — Total advertising expenditure by vehicle type during recession


In [ ]:
vehicle_ad = recession_data.groupby(
    "Vehicle_Type"
)["Advertising_Expenditure"].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 8))
plt.pie(
    vehicle_ad.values,
    labels=vehicle_ad.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Share of Total Advertising Expenditure by Vehicle Type during Recession")
plt.tight_layout()
plt.savefig("Task_1_8_Advertising_by_Vehicle_Type_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Task 1.9 — Effect of unemployment rate on vehicle type and sales during recession


In [ ]:
unemployment_vehicle = recession_data.groupby(
    "Vehicle_Type", as_index=False
).agg(
    unemployment_rate=("unemployment_rate", "mean"),
    Automobile_Sales=("Automobile_Sales", "mean")
)

fig, ax1 = plt.subplots(figsize=(11, 6))
x = np.arange(len(unemployment_vehicle))
width = 0.38

ax1.bar(
    x - width / 2,
    unemployment_vehicle["Automobile_Sales"],
    width,
    label="Average Sales"
)
ax1.set_ylabel("Average Automobile Sales")
ax1.set_xticks(x)
ax1.set_xticklabels(unemployment_vehicle["Vehicle_Type"], rotation=25)

ax2 = ax1.twinx()
ax2.plot(
    x,
    unemployment_vehicle["unemployment_rate"],
    marker="o",
    linewidth=2,
    label="Unemployment Rate"
)
ax2.set_ylabel("Average Unemployment Rate")

plt.title("Unemployment Rate and Vehicle Sales by Vehicle Type during Recession")
fig.tight_layout()
plt.savefig("Task_1_9_Unemployment_vs_Vehicle_Sales_Recession.png",
            dpi=150, bbox_inches="tight")
plt.show()


## Final step before submission

Run **Kernel → Restart & Run All** (or **Run All**) before uploading so every output is embedded in the `.ipynb` file. The plotting cells also save PNG files using the task numbers.
